# Train model

In [1]:
import graphein
import lightning as L
import pandas as pd
import torch
import torch.nn.functional as F
from lightning.pytorch.callbacks.early_stopping import EarlyStopping
from optimizer.sam import SAM
from torch import nn
from torch_geometric.loader import DataLoader

# Set constants

In [2]:
# Training and validation dataset paths to train and stop models
training_data_path = "data/"

SEED = 255
L.seed_everything(SEED)

# Load raw data and untrained model

## Training data

## Define and init model class

In [3]:
class DenseLayer(nn.Module):
    """Convinience layer for combinding linear, leaky_relu, and
    dropout into one Module
    """

    def __init__(
        self,
        in_layers,
        out_layers,
        negative_slope=0.1,
        p=0.3,
    ):
        super().__init__()

        self.layer = nn.Sequential(
            nn.Linear(in_layers, out_layers),
            nn.LeakyReLU(negative_slope),
            nn.Dropout(p),
        )

    def forward(self, x):
        return self.layer(x)

In [4]:
class AbAffinityPredictionModel(nn.Module):
    def __init__(self):
        super().__init__()

        # X

        # Regression head
        self.regression_head = nn.Sequential(
            DenseLayer(10, 5),
            DenseLayer(5, 3),
            nn.Linear(3, 1),
        )

    def forward(self, x):
        y_pred = self.regression_head(x)
        return y_pred

NOTE: model trained using hyperparam from hyperparam_opt

In [5]:
model = AbAffinityPredictionModel()
model.eval()

AbAffinityPredictionModel(
  (regression_head): Sequential(
    (0): DenseLayer(
      (layer): Sequential(
        (0): Linear(in_features=10, out_features=5, bias=True)
        (1): LeakyReLU(negative_slope=0.1)
        (2): Dropout(p=0.3, inplace=False)
      )
    )
    (1): DenseLayer(
      (layer): Sequential(
        (0): Linear(in_features=5, out_features=3, bias=True)
        (1): LeakyReLU(negative_slope=0.1)
        (2): Dropout(p=0.3, inplace=False)
      )
    )
    (2): Linear(in_features=3, out_features=1, bias=True)
  )
)

In [6]:
x = torch.Tensor(range(10))
model(x)

tensor([0.7152], grad_fn=<ViewBackward0>)

In [7]:
# class LitAutoEncoder(L.LightningModule):
#     def __init__(self, encoder, decoder):
#         super().__init__()
#         self.encoder = encoder
#         self.decoder = decoder

#     def training_step(self, batch, batch_idx):
#         # training_step defines the train loop.
#         x, _ = batch
#         x = x.view(x.size(0), -1)
#         z = self.encoder(x)
#         x_hat = self.decoder(z)
#         loss = F.mse_loss(x_hat, x)
#         return loss

#     def configure_optimizers(self):
#         optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
#         return optimizer

In [8]:
class LitAbAffinityPredictionModel(L.LightningModule):
    def __init__(self, affinity_model: AbAffinityPredictionModel):
        super().__init__()
        self.model = affinity_model

    def forward(self, x):
        return self.model(x)

    def _generate_loss(self, batch, batch_idx):
        x, y = batch.x, batch.graph_y
        y_pred = self.affinity_model(x)
        loss = F.mse_loss(y_pred, y)
        return loss

    def training_step(self, batch, batch_idx):
        # training_step defines the train loop.
        loss = self._generate_loss(batch, batch_idx)
        return loss

    def validation_step(self, batch, batch_idx):
        # this is the validation loop
        val_loss = self._generate_loss(batch, batch_idx)
        self.log("val_loss", val_loss)

    def test_step(self, batch, batch_idx):
        # training_step defines the train loop.
        test_loss = self._generate_loss(batch, batch_idx)
        self.log("test_loss", test_loss)

    def configure_optimizers(self):
        optimizer = torch.optim.Adam(self.parameters(), lr=1e-3)
        # optimizer = SAM()
        return optimizer

In [9]:
lightning_model = LitAbAffinityPredictionModel(model)

In [10]:
lightning_model(x)

tensor([0.7152], grad_fn=<ViewBackward0>)

# Preprocess data

In [11]:
# Define X

In [12]:
trainer = L.Trainer()
help(trainer.test)

/home/bobby/miniconda3/lib/python3.13/site-packages/lightning/fabric/plugins/environments/slurm.py:204: The `srun` command is available on your system but is not used. HINT: If your intention is to run Lightning on SLURM, prepend your python command with `srun` like so: srun python /home/bobby/miniconda3/lib/python3.13/site-packages/ ...
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores


Help on method test in module lightning.pytorch.trainer.trainer:

test(
    model: Optional[ForwardRef('pl.LightningModule')] = None,
    dataloaders: Union[Any, lightning.pytorch.core.datamodule.LightningDataModule, NoneType] = None,
    ckpt_path: Union[str, pathlib._local.Path, NoneType] = None,
    verbose: bool = True,
    datamodule: Optional[lightning.pytorch.core.datamodule.LightningDataModule] = None
) -> list[collections.abc.Mapping[str, float]] method of lightning.pytorch.trainer.trainer.Trainer instance
    Perform one evaluation epoch over the test set. It's separated from fit to make sure you never run on your
    test set until you want to.

    Args:
        model: The model to test.

        dataloaders: An iterable or collection of iterables specifying test samples.
            Alternatively, a :class:`~lightning.pytorch.core.datamodule.LightningDataModule` that defines
            the :class:`~lightning.pytorch.core.hooks.DataHooks.test_dataloader` hook.

        ckp

In [13]:
# DataLoader

In [14]:
# x_train --> x_train, x_val

train_set_size = int(len(train_set) * 0.8)
valid_set_size = len(train_set) - train_set_size

# split the train set into two
# seed = torch.Generator().manual_seed(SEED)
# train_set, valid_set = data.random_split(train_set, [train_set_size, valid_set_size], generator=seed)
# train_loader, valid_loader)

NameError: name 'train_set' is not defined

In [ ]:
# train_loader = DataLoader
# val data_loader = DataLoader

# Train model

In [ ]:
# # model
# autoencoder = LitAutoEncoder(Encoder(), Decoder())

# # train model
# trainer = L.Trainer()
# trainer.fit(model=autoencoder, train_dataloaders=train_loader)

In [ ]:
# callbacks=[EarlyStopping(monitor="val_loss", mode="min")]

In [ ]:
# Trainer
trainer = L.Trainer()
# trainer.fit(model, train_loader, valid_loader)

In [ ]:
# Once the model has finished training, call .test
# trainer.test(model, dataloaders=DataLoader(test_set))